In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path(r"C:\Users\Administrator\EDIT-1-24012456\data\E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("../data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请检查文件存放路径")

# 项目输出目录
OUTPUT_DIR = Path("output/day04_project")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 读取原始数据
raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据文件：{DATA_PATH}")
print(f"输出目录：{OUTPUT_DIR.resolve()}")
print(f"原始数据形状：{raw_df.shape[0]} 行 , {raw_df.shape[1]} 列")
raw_df.head()
# 1.每条记录代表什么？
# 单条电商平台用户完整画像，包含会员时长、设备、支付、订单、流失状态等全维度行为数据
# 2.项目的目标变量是哪一列？
# Churn：1代表用户流失，0代表用户留存，是流失预测目标字段
# 3.为什么 CustomerID 不应作为普通连续数值参与后续分析？
# CustomerID仅为用户唯一标识，无数值大小意义，不能做加减运算，属于分类唯一编号，直接当数值会误导模型
def build_quality_report(data):
    """返回字段级数据质量报告：类型、缺失数量、缺失比例、唯一值数量"""
    report = pd.DataFrame()
    report["字段类型"] = data.dtypes
    report["缺失数量"] = data.isna().sum()
    report["缺失比例(%)"] = round(data.isna().mean() * 100, 2)
    report["唯一值数量"] = data.nunique()
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)
# 完全重复整行
total_dup = raw_df.duplicated().sum()
# 用户ID重复
cid_dup = raw_df["CustomerID"].duplicated().sum()
# 流失分布
churn_cnt = raw_df["Churn"].value_counts()
churn_rate = round(raw_df["Churn"].mean() * 100, 2)

print("完全重复行数：", total_dup)
print("CustomerID 重复数量：", cid_dup)
print("\nChurn 流失分布：")
print(churn_cnt)
print(f"整体用户流失率：{churn_rate}%")

# 分类字段频数
cat_list = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
for col in cat_list:
    print(f"\n===== {col} =====")
    print(raw_df[col].value_counts())
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone",
        "Mobile": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。
    参数：data: 原始用户行为 DataFrame
    返回：cleaned_df: 清洗后数据集；cleaning_log: 处理日志表
    """
    # 复制数据，不修改原始df
    df = data.copy()
    logs = []
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 步骤1：删除完全重复行
    before_row = df.shape[0]
    df = df.drop_duplicates()
    after_row = df.shape[0]
    affect = before_row - after_row
    logs.append({
        "处理时间": now,
        "步骤": "删除完全重复行",
        "规则": "移除完全一致的重复记录",
        "处理前行数": before_row,
        "处理后行数": after_row,
        "影响记录数": affect
    })

    # 步骤2：数值字段中位数填充缺失
    for col in NUMERIC_MISSING_COLS:
        miss_before = df[col].isna().sum()
        med_val = df[col].median()
        df[col] = df[col].fillna(med_val)
        miss_after = df[col].isna().sum()
        affect = miss_before - miss_after
        logs.append({
            "处理时间": now,
            "步骤": f"中位数填充-{col}",
            "规则": f"使用字段中位数 {med_val:.2f} 填补缺失",
            "处理前行数": miss_before,
            "处理后行数": miss_after,
            "影响记录数": affect
        })

    # 步骤3：分类字段统一名称
    for col, map_dict in CATEGORY_MAPPINGS.items():
        for old_val, new_val in map_dict.items():
            cnt = (df[col] == old_val).sum()
            df[col] = df[col].replace(old_val, new_val)
            logs.append({
                "处理时间": now,
                "步骤": f"类别标准化-{col}",
                "规则": f"将 {old_val} 统一映射为 {new_val}",
                "处理前行数": cnt,
                "处理后行数": 0,
                "影响记录数": cnt
            })

    # 步骤4：目标字段转为整型
    df["Churn"] = df["Churn"].astype(int)
    df["Complain"] = df["Complain"].astype(int)

    # 生成日志DataFrame
    cleaning_log = pd.DataFrame(logs)
    return df, cleaning_log

# 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
display(cleaning_log)
cleaned_df.head()
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "下限": round(lower, 2),
        "上限": round(upper, 2),
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# 1. 新增用户时长分层 TenureGroup
bins = [0, 12, 24, 36, 1000]
labels = ["0-12个月", "13-24个月", "25-36个月", "36个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=bins, labels=labels)

# 2. 新增移动端登录标记 IsMobileLogin
mobile_tags = ["Mobile Phone"]
cleaned_df["IsMobileLogin"] = np.where(cleaned_df["PreferredLoginDevice"].isin(mobile_tags), 1, 0)

# 3. 生成异常值报告
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = []
for c in outlier_cols:
    res = iqr_outlier_summary(cleaned_df[c])
    res["字段名"] = c
    outlier_report.append(res)
outlier_report = pd.DataFrame(outlier_report)
print("===== IQR候选异常值报告 =====")
display(outlier_report)
# 统计不合规数据
rule_data = [
    ["使用时长小于0", (cleaned_df["Tenure"] < 0).sum()],
    ["仓库距离小于0", (cleaned_df["WarehouseToHome"] < 0).sum()],
    ["订单数小于或等于0", (cleaned_df["OrderCount"] <= 0).sum()],
    ["返现金额小于0", (cleaned_df["CashbackAmount"] < 0).sum()]
]
business_rule_report = pd.DataFrame(rule_data, columns=["规则", "不合规记录数"])
display(business_rule_report)

# 处理结论
# 本数据集无负数违规业务数据，无需删除/修正；若存在负数，需和业务确认是录入错误还是特殊标记，不可直接丢弃
# 清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 验收断言
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍存在缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "登录设备未完成标准化"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式简写未清理"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式简写未清理"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "衍生特征缺失"

# 导出4份交付文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

print("✅ 全部校验通过！")
print(f"清洗前质量报告：{OUTPUT_DIR / 'data_quality_before.csv'}")
print(f"清洗后质量报告：{OUTPUT_DIR / 'data_quality_after.csv'}")
print(f"数据处理日志：{OUTPUT_DIR / 'cleaning_log.csv'}")
print(f"最终清洗数据集：{OUTPUT_DIR / 'ecommerce_customer_cleaned.csv'}")
print("\n异常值报告：")
display(outlier_report)
print("\n业务规则校验报告：")
display(business_rule_report)
#数据问题：数值字段存在缺失、分类名称不统一、存在统计学极值异常值；
#处理策略：缺失值用中位数填充、同义类别统一命名、异常值仅记录不删除；
#可用于后续分析：无缺失、分类标准化、新增分层特征，数据干净无噪声；
#需业务确认：IQR 筛选出的极值是否为真实高价值用户，负数业务阈值定义。